In [100]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

# Load the dataset
df = pd.read_csv('Data set for DADS June.csv')
print(df.head())


          Date   Time                       Description   Type  Amount  \
0   2024-01-01  03:11                AMAZON SELLER SVCS  Debit   ₹2462   
1    01-Jan-24  05:44                         BHIM-BMTC     DR   50.00   
2    01-Jan-24  09:35  NEFT-TECHCRUSH LABS-SALARY MAY24     CR  ₹84728   
3   2024-01-01  14:07              UPI-AMAN-8934@OKAXIS  Debit   ₹1828   
4  01 Jan 2024  14:23                      BHIM-BLINKIT  Debit  270.00   

    Balance  Mode        Ref  
0  678275.0   UPI  TXN190872  
1  681007.0   UPI  TXN143064  
2  484728.0  NEFT  TXN246316  
3 -748745.0   UPI  TXN569226  
4  680737.0   UPI  TXN968962  


In [101]:
def clean_data(df):
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)
    df['hour'] = df['Time'].str.split(':').str[0].astype(int)
    df['month'] = df['Date'].dt.month
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['week_number'] = df['Date'].dt.isocalendar().week

    df["Amount"] = (
        df["Amount"]
        .astype(str)
        .str.replace("₹", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("Rs.", "", regex=False)
        .str.strip()
    )
    df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
    
    df['Type'] = df['Type'].str.strip().str.lower()
    df['Type'] = df['Type'].map({
        "dr": "debit",
        "cr": "credit",
        "debit": "debit",
        "credit": "credit"
    })
    df["Mode"] = df["Mode"].astype(str).str.strip()
    df["Mode"] = df["Mode"].replace("", np.nan)
    
    df = df.dropna(subset=['Date', 'Time', 'Amount', 'Type'])
    df = df.drop_duplicates()
    
    return df

df = clean_data(df)
print(f"\nMissing values:\n{df.isnull().sum()}")




Missing values:
Date           0
Time           0
Description    0
Type           0
Amount         0
Balance        0
Mode           0
Ref            0
hour           0
month          0
day_of_week    0
week_number    0
dtype: int64


Idk why its 0

In [102]:
# Define vendor patterns
vendors = {
    # Food & Grocery
    r'INSTAMART': 'SWIGGY INSTAMART',
    r'SWIGGY': 'SWIGGY',
    r'ZEPTO': 'ZEPTO',
    r'ZOMATO': 'ZOMATO',
    r'BLINKIT': 'BLINKIT',
    r'GROFERS': 'GROFERS',
    r'BUNDL': 'BUNDL',
    r'KIRANAKART': 'KIRANAKART',
    r'DMART': 'DMART',
    r'BIGBASKET': 'BIGBASKET',
    
    # Shopping
    r'AMAZON': 'AMAZON',
    r'FLIPKART': 'FLIPKART',
    r'MYNTRA': 'MYNTRA',
    r'NYKAA': 'NYKAA',
    r'AMZN': 'AMZN',
    
    # Transport
    r'OLA': 'OLA',
    r'UBER': 'UBER',
    r'RAPIDO': 'RAPIDO',
    r'BMTC': 'BMTC BUS TRANSIT',
    
    # Cafes & Restaurants
    r'STARBUCKS': 'STARBUCKS',
    r'THIRDWAVE': 'THIRD WAVE',
    r'CCD': 'CAFE COFFEE DAY',
    r'COFFEE DAY': 'COFFEE DAY',
    r'TRUFFLES': 'TRUFFLES',
    r'MEGHANA': 'MEGHANA',
    
    # Entertainment & Subscriptions
    r'NETFLIX': 'NETFLIX',
    r'HOTSTAR': 'HOTSTAR',
    r'SPOTIFY': 'SPOTIFY',
    r'BOOKMYSHOW': 'BOOKMYSHOW',
    r'PRIME VIDEO': 'PRIME VIDEO',
    
    # Investments
    r'ZERODHA': 'ZERODHA',
    r'GROWW': 'GROWW',
    
    # Utilities & Bills
    r'BESCOM': 'BESCOM',
    r'BWSSB': 'BWSSB',
    r'AIRTEL': 'AIRTEL',
    r'JIO': 'JIO',
    r'INDIAN OIL': 'INDIAN OIL',
    r'BPCL': 'BPCL',
    r'HP PETROL': 'HP PETROL',
    
    # Banking
    r'ATM-WDL': 'ATM WITHDRAWAL',
    r'SALARY': 'SALARY INCOME',
    r'UPI-[A-Z]+-\d+@|UPI-[A-Z]+@': 'PEER TO PEER'
}

def extract_vendor(text):
    for pattern, vendor_name in vendors.items():
        if re.search(pattern, str(text), re.IGNORECASE):
            return vendor_name
    return 'OTHER / UNKNOWN'

df['Extracted_Vendor'] = df['Description'].apply(extract_vendor)
df['Extracted_Vendor'].unique()


<StringArray>
[          'AMAZON',     'PEER TO PEER',            'ZEPTO',
             'UBER',           'SWIGGY',  'OTHER / UNKNOWN',
        'STARBUCKS',          'BLINKIT',           'ZOMATO',
              'OLA',            'DMART',           'MYNTRA',
 'SWIGGY INSTAMART',            'NYKAA',         'FLIPKART',
          'ZERODHA',       'THIRD WAVE',    'SALARY INCOME',
           'RAPIDO',       'COFFEE DAY',  'CAFE COFFEE DAY',
       'BOOKMYSHOW',          'HOTSTAR',          'NETFLIX',
             'AMZN',        'BIGBASKET',        'HP PETROL',
       'KIRANAKART',   'ATM WITHDRAWAL',            'GROWW']
Length: 30, dtype: str

In [103]:
print(f"\nTop 10 vendors:\n{df['Extracted_Vendor'].value_counts().head(10)}")



Top 10 vendors:
Extracted_Vendor
OTHER / UNKNOWN     23
SWIGGY              22
ZOMATO              16
SWIGGY INSTAMART     9
OLA                  8
UBER                 7
PEER TO PEER         6
ZEPTO                5
STARBUCKS            4
NYKAA                4
Name: count, dtype: int64


In [104]:
def assign_expense_category(vendor):
    vendor = str(vendor).upper()
    
    categories = {
        "Food & Dining": [
            "SWIGGY", "ZOMATO", "BUNDL", "STARBUCKS", 
            "THIRD WAVE", "CAFE COFFEE DAY", "TRUFFLES", "MEGHANA"
        ],
        "Quick Commerce": [
            "SWIGGY INSTAMART", "ZEPTO", "KIRANAKART", "BLINKIT", 
            "GROFERS", "BIGBASKET", "DMART"
        ],
        "Shopping": [
            "AMAZON", "FLIPKART", "MYNTRA", "NYKAA", "AMZN"
        ],
        "Transport": [
            "OLA", "UBER", "RAPIDO", "BMTC BUS TRANSIT"
        ],
        "Entertainment": [
            "NETFLIX", "HOTSTAR", "SPOTIFY", "BOOKMYSHOW", "PRIME VIDEO"
        ],
        "Investments": [
            "ZERODHA", "GROWW"
        ],
        "Utilities & Bills": [
            "BESCOM", "BWSSB", "AIRTEL", "JIO", "INDIAN OIL", "BPCL", "HP PETROL"
        ],
        "Banking": [
            "ATM WITHDRAWAL", "SALARY INCOME", "PEER TO PEER"
        ]
    }
    
    for category, keywords in categories.items():
        if any(keyword in vendor for keyword in keywords):
            return category
    
    return "Others"

df['Expense_Category'] = df['Extracted_Vendor'].apply(assign_expense_category)

df['Expense_Category'].unique()



<StringArray>
[         'Shopping',           'Banking',    'Quick Commerce',
         'Transport',     'Food & Dining',            'Others',
       'Investments',     'Entertainment', 'Utilities & Bills']
Length: 9, dtype: str

In [105]:
print("\nCategory distribution:\n",df['Expense_Category'].value_counts())


Category distribution:
 Expense_Category
Food & Dining        54
Others               24
Transport            17
Shopping             13
Quick Commerce       13
Banking              11
Investments           5
Entertainment         5
Utilities & Bills     1
Name: count, dtype: int64


In [106]:
# Total credits
credit_df = df[df["Type"] == "credit"]
total_credits = credit_df["Amount"].sum()

# Total debits
debit_df = df[df["Type"] == "debit"]
total_debits = debit_df["Amount"].sum()

# Net amount
net_amount = total_credits - total_debits

# Savings rate
if total_credits > 0:
    savings_rate = (net_amount / total_credits) * 100
else:
    savings_rate = 0

# Other details
total_transactions = len(df)
unique_vendors = df["Extracted_Vendor"].nunique()

start_date = df["Date"].min().date()
end_date = df["Date"].max().date()

print("=" * 60)
print("FINANCIAL SUMMARY")
print("=" * 60)

print("Total Income (Credits)    :", round(total_credits, 2))
print("Total Expenses (Debits)   :", round(total_debits, 2))
print("Net Amount                :", round(net_amount, 2))
print("Savings Rate              :", round(savings_rate, 2), "%")
print("Total Transactions        :", total_transactions)
print("Unique Vendors            :", unique_vendors)
print("Transaction Period        :", start_date, "to", end_date)

FINANCIAL SUMMARY
Total Income (Credits)    : 170094.0
Total Expenses (Debits)   : 205310.0
Net Amount                : -35216.0
Savings Rate              : -20.7 %
Total Transactions        : 143
Unique Vendors            : 30
Transaction Period        : 2024-01-01 to 2024-12-06


In [107]:
debit_df = df[df["Type"] == "debit"]
category_total = debit_df.groupby("Expense_Category")["Amount"].sum()
category_total = category_total.sort_values(ascending=False)

print("TOP 5 CATEGORIES")
print(category_total.head(5))
vendor_total = debit_df.groupby("Extracted_Vendor")["Amount"].sum()
vendor_total = vendor_total.sort_values(ascending=False)

print("\nTOP 5 VENDORS")

for vendor, amount in vendor_total.head(5).items():
    transactions = debit_df[debit_df["Extracted_Vendor"] == vendor].shape[0]
    print(vendor, amount, transactions)

TOP 5 CATEGORIES
Expense_Category
Investments      64496.0
Others           52994.0
Shopping         36416.0
Food & Dining    23592.0
Banking           9300.0
Name: Amount, dtype: float64

TOP 5 VENDORS
ZERODHA 60000.0 4
OTHER / UNKNOWN 52753.0 23
MYNTRA 13629.0 2
FLIPKART 10041.0 4
SWIGGY 9845.0 22


In [108]:
debit_df = df[df["Type"] == "debit"]

categories = debit_df["Expense_Category"].unique()
months = sorted(debit_df["month"].unique())

monthly_trend = debit_df.pivot_table(
    values='Amount',
    index='Expense_Category',
    columns='month',
    aggfunc='sum',
    fill_value=0
)

print("\nMONTHLY SPENDING TREND BY CATEGORY\n")

for category in categories:

    print(category)

    category_df = debit_df[debit_df["Expense_Category"] == category]

    for month in months:
        amount = category_df[category_df["month"] == month]["Amount"].sum()
        print("Month", month, ":", amount)

    print()

if len(months) > 1:

    first = months[0]
    last = months[-1]

    print("Growth from Month", first, "to Month", last)

    for category in categories:

        category_df = debit_df[debit_df["Expense_Category"] == category]

        first_amount = category_df[category_df["month"] == first]["Amount"].sum()
        last_amount = category_df[category_df["month"] == last]["Amount"].sum()

        if first_amount == 0:
            growth = 0
        else:
            growth = ((last_amount - first_amount) / first_amount) * 100

        print(category, ":", round(growth, 1), "%")


MONTHLY SPENDING TREND BY CATEGORY

Shopping
Month 1.0 : 6972.0
Month 2.0 : 996.0
Month 3.0 : 2147.0
Month 4.0 : 3311.0
Month 5.0 : 0.0
Month 6.0 : 4868.0
Month 7.0 : 2425.0
Month 8.0 : 860.0
Month 9.0 : 10745.0
Month 10.0 : 0.0
Month 11.0 : 0.0
Month 12.0 : 4092.0

Banking
Month 1.0 : 1828.0
Month 2.0 : 0.0
Month 3.0 : 501.0
Month 4.0 : 0.0
Month 5.0 : 0.0
Month 6.0 : 1860.0
Month 7.0 : 0.0
Month 8.0 : 0.0
Month 9.0 : 246.0
Month 10.0 : 599.0
Month 11.0 : 2500.0
Month 12.0 : 1766.0

Quick Commerce
Month 1.0 : 825.0
Month 2.0 : 697.0
Month 3.0 : 1382.0
Month 4.0 : 0.0
Month 5.0 : 502.0
Month 6.0 : 352.0
Month 7.0 : 1453.0
Month 8.0 : 1862.0
Month 9.0 : 527.0
Month 10.0 : 0.0
Month 11.0 : 347.0
Month 12.0 : 910.0

Transport
Month 1.0 : 826.0
Month 2.0 : 621.0
Month 3.0 : 439.0
Month 4.0 : 0.0
Month 5.0 : 431.0
Month 6.0 : 134.0
Month 7.0 : 351.0
Month 8.0 : 366.0
Month 9.0 : 710.0
Month 10.0 : 1232.0
Month 11.0 : 35.0
Month 12.0 : 0.0

Food & Dining
Month 1.0 : 2266.0
Month 2.0 : 3809.

In [109]:
hourly_trend = debit_df.pivot_table(
    values='Amount',
    index='Expense_Category',
    columns='hour',
    aggfunc='sum',
    fill_value=0
)

print("\n" + "="*70)
print("PEAK SPENDING HOURS BY CATEGORY")
print("="*70)
for category in hourly_trend.index:
    peak_hour = hourly_trend.loc[category].idxmax()
    peak_amount = hourly_trend.loc[category].max()
    print(f"{category:<30} Peak: {peak_hour:02d}:00 → ₹{peak_amount:>10,.2f}")
print("\n" + "="*70)
print("LATE NIGHT SPENDING ANALYSIS (21:00 - 02:00)")
print("="*70)
late_night_df = debit_df[(debit_df['hour'] >= 21) | (debit_df['hour'] <= 2)]
late_categories = late_night_df.groupby('Expense_Category')['Amount'].agg(['sum', 'count'])
late_categories.columns = ['Total Amount', 'Transactions']
late_categories = late_categories.sort_values('Total Amount', ascending=False)
print(late_categories)



PEAK SPENDING HOURS BY CATEGORY
Banking                        Peak: 10:00 → ₹  2,599.00
Entertainment                  Peak: 14:00 → ₹  1,473.00
Food & Dining                  Peak: 12:00 → ₹  2,791.00
Investments                    Peak: 10:00 → ₹ 45,000.00
Others                         Peak: 13:00 → ₹ 20,255.00
Quick Commerce                 Peak: 11:00 → ₹  3,075.00
Shopping                       Peak: 20:00 → ₹ 10,745.00
Transport                      Peak: 15:00 → ₹    908.00
Utilities & Bills              Peak: 19:00 → ₹  1,087.00

LATE NIGHT SPENDING ANALYSIS (21:00 - 02:00)
                  Total Amount  Transactions
Expense_Category                            
Food & Dining           5742.0            12
Shopping                4153.0             2
Banking                 3126.0             2
Transport               1137.0             4
Others                  1053.0             3
Entertainment            905.0             2
Quick Commerce           697.0             1


In [110]:

debit_df = df[df["Type"] == "debit"].copy()
debit_df["category_mean"] = debit_df.groupby("Expense_Category")["Amount"].transform("mean")
debit_df["category_std"] = debit_df.groupby("Expense_Category")["Amount"].transform("std")

debit_df["z_score"] = (
    (debit_df["Amount"] - debit_df["category_mean"])
    / debit_df["category_std"].replace(0, 1)
)

anomalies = debit_df[abs(debit_df["z_score"]) > 2]

anomalies = anomalies.sort_values("z_score", ascending=False)

print("\n" + "=" * 70)
print(f"ANOMALY DETECTION (Z-score > 2) - {len(anomalies)} Transactions")
print("=" * 70)

if len(anomalies) > 0:
    print(
        anomalies[
            [
                "Date",
                "Time",
                "Extracted_Vendor",
                "Expense_Category",
                "Amount",
                "z_score",
            ]
        ].head(15).to_string(index=False)
    )
else:
    print("No anomalies detected!")


ANOMALY DETECTION (Z-score > 2) - 5 Transactions
      Date  Time Extracted_Vendor Expense_Category  Amount  z_score
2024-03-06 14:21  OTHER / UNKNOWN           Others 18000.0 3.216815
2024-05-05 13:23  OTHER / UNKNOWN           Others 18000.0 3.216815
2024-09-01 20:00           MYNTRA         Shopping 10745.0 3.072370
2024-01-02 09:20 SWIGGY INSTAMART    Food & Dining   791.0 2.504209
2024-08-01 11:44            DMART   Quick Commerce  1862.0 2.334431


In [111]:
forecast = {}

for category in monthly_trend.index:

    amounts = list(monthly_trend.loc[category])

    if len(amounts) >= 3:
        avg = (amounts[-1] + amounts[-2] + amounts[-3]) / 3
    else:
        avg = sum(amounts) / len(amounts)

    forecast[category] = avg

forecast_df = pd.DataFrame(forecast.items())
forecast_df.columns = ["Expense_Category", "Predicted Next Month Spend"]

forecast_df = forecast_df.sort_values(
    by="Predicted Next Month Spend",
    ascending=False
)

print("\n" + "=" * 60)
print("NEXT MONTH SPENDING PREDICTION")
print("=" * 60)

print(forecast_df.round(2).to_string(index=False))

total = forecast_df["Predicted Next Month Spend"].sum()
print("\nPredicted Total Spending: ₹", round(total, 2))


NEXT MONTH SPENDING PREDICTION
 Expense_Category  Predicted Next Month Spend
      Investments                     5000.00
           Others                     2988.67
    Food & Dining                     1761.00
          Banking                     1621.67
         Shopping                     1364.00
        Transport                      422.33
   Quick Commerce                      419.00
Utilities & Bills                      362.33
    Entertainment                      278.33

Predicted Total Spending: ₹ 14217.33


In [112]:
debit_df = df[df["Type"] == "debit"]
category_spending = debit_df.groupby("Expense_Category")["Amount"].sum()

profile = {}

food = category_spending.get("Food & Dining", 0)
quick = category_spending.get("Quick Commerce", 0)

food_percent = ((food + quick) / total_debits) * 100

if food_percent > 15:
    profile["THE FOODIE"] = True
else:
    profile["THE FOODIE"] = False

shopping = category_spending.get("Shopping", 0)
shopping_percent = (shopping / total_debits) * 100

if shopping_percent > 15:
    profile["THE SHOPAHOLIC"] = True
else:
    profile["THE SHOPAHOLIC"] = False


transport = category_spending.get("Transport", 0)
transport_percent = (transport / total_debits) * 100

if transport_percent >= 10:
    profile["THE CAB COMMUTER"] = True
else:
    profile["THE CAB COMMUTER"] = False

entertainment = category_spending.get("Entertainment", 0)
entertainment_percent = (entertainment / total_debits) * 100

if entertainment_percent >= 5:
    profile["THE ENTERTAINMENT JUNKIE"] = True
else:
    profile["THE ENTERTAINMENT JUNKIE"] = False

investment = category_spending.get("Investments", 0)
investment_percent = (investment / total_debits) * 100

if investment_percent > 5:
    profile["THE INVESTOR"] = True
else:
    profile["THE INVESTOR"] = False

if total_credits > 0:
    savings_rate = ((total_credits - total_debits) / total_credits) * 100
else:
    savings_rate = 0

if savings_rate > 40:
    profile["THE DISCIPLINED SAVER"] = True
else:
    profile["THE DISCIPLINED SAVER"] = False

if savings_rate < 10:
    profile["THE YOLO SPENDER"] = True
else:
    profile["THE YOLO SPENDER"] = False

others = category_spending.get("Others", 0)
others_percent = (others / total_debits) * 100

if others_percent > 10:
    profile["THE MISCELLANEOUS SPENDER"] = True
else:
    profile["THE MISCELLANEOUS SPENDER"] = False

print("\nYOUR SPENDING PERSONALITY PROFILE\n")

for name in profile:
    print(name, "-", profile[name])

print("\nCATEGORY SPENDING BREAKDOWN\n")

for category, amount in category_spending.items():
    percent = (amount / total_debits) * 100
    print(category, "-", round(percent, 1), "%", "- ₹", round(amount, 2))

print("\nSavings Rate:", round(savings_rate, 1), "%")


YOUR SPENDING PERSONALITY PROFILE

THE FOODIE - True
THE SHOPAHOLIC - True
THE CAB COMMUTER - False
THE ENTERTAINMENT JUNKIE - False
THE INVESTOR - True
THE DISCIPLINED SAVER - False
THE YOLO SPENDER - True
THE MISCELLANEOUS SPENDER - True

CATEGORY SPENDING BREAKDOWN

Banking - 4.5 % - ₹ 9300.0
Entertainment - 1.7 % - ₹ 3423.0
Food & Dining - 11.5 % - ₹ 23592.0
Investments - 31.4 % - ₹ 64496.0
Others - 25.8 % - ₹ 52994.0
Quick Commerce - 4.3 % - ₹ 8857.0
Shopping - 17.7 % - ₹ 36416.0
Transport - 2.5 % - ₹ 5145.0
Utilities & Bills - 0.5 % - ₹ 1087.0

Savings Rate: -20.7 %


In [113]:
print("\n" + "=" * 70)
print("DETAILED EXPENSE ANALYSIS")
print("=" * 70)

debit_analysis = debit_df[
    ["Date", "Time", "Extracted_Vendor", "Expense_Category",
     "Amount", "Mode", "Type"]
]

debit_analysis = debit_analysis.sort_values("Date")

print(debit_analysis)

print("\n" + "=" * 70)
print("CATEGORY SUMMARY")
print("=" * 70)

category_summary = debit_df.groupby("Expense_Category")["Amount"]

total = category_summary.sum()
average = category_summary.mean()
count = category_summary.count()

for category in total.index:
    print("\nCategory :", category)
    print("Total Spending       :", round(total[category], 2))
    print("Average Transaction  :", round(average[category], 2))
    print("Transaction Count    :", count[category])

print("\nAnalysis completed successfully!")


DETAILED EXPENSE ANALYSIS
           Date   Time Extracted_Vendor Expense_Category  Amount Mode   Type
0    2024-01-01  03:11           AMAZON         Shopping  2462.0  UPI  debit
3    2024-01-01  14:07     PEER TO PEER          Banking  1828.0  UPI  debit
5    2024-01-01  14:48            ZEPTO   Quick Commerce   625.0  UPI  debit
6    2024-01-01  14:50             UBER        Transport   148.0  UPI  debit
225  2024-01-02  23:34           ZOMATO    Food & Dining   606.0  UPI  debit
...         ...    ...              ...              ...     ...  ...    ...
964  2024-12-05  20:55           SWIGGY    Food & Dining   615.0  UPI  debit
1195 2024-12-06  14:38           SWIGGY    Food & Dining   368.0  UPI  debit
1193 2024-12-06  09:29   ATM WITHDRAWAL          Banking   500.0  ATM  debit
1194 2024-12-06  09:59           ZOMATO    Food & Dining   383.0  UPI  debit
1197 2024-12-06  16:29           ZOMATO    Food & Dining   377.0  UPI  debit

[141 rows x 7 columns]

CATEGORY SUMMARY

Catego